# Agent evaluation (Phases 9–10): repair agent vs agent baselines (GPU)

Runs agent systems over the R / C / T scenarios (`eval/agent_scenarios/rct_v2.jsonl`, built from pool v3: all four datasets) and the E / S scenarios (`eval/agent_scenarios/{E,S}/`) with one
real model (same model, limits and budget for every system), and judges each final output against the reference.
Scoring runs on the laptop: `python eval/agent_metrics.py results/agent_v1`.

**Step 1 (`STAGE = "dev"`):** full agent on the dev scenarios only (14 R + 20 C + 6 T + 3 E + 3 S), to check the agent prompt with the real model
(README Phase 9, task 5). **Step 2 (`STAGE = "test"`):** all systems on the test scenarios.
Everything is resumable and cached; outputs per system go to `agent_<system>.zip`.

Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.

**GPU time (estimate):** about 15 s per agent step for the 7B in 4-bit, so ~3 h per system on test (83 R + 24 T + 15 E/S; C is nearly free with the oracle verifier). Run one system per session if needed: it resumes.

In [1]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True     # one model for every agent system
STAGE = "dev"                 # "dev" first (prompt check), then "test"
VERIFIER = "oracle"           # oracle | parsed:<path to parses.jsonl> | conventional
# (system name, mode, extra flags). Main systems first; ablations are optional (R scenarios only, to save GPU time).
SYSTEMS = [
    ("agent_full",     "full",     []),
    ("agent_feedback", "feedback", []),
    ("agent_naive",    "naive",    []),
]
ABLATIONS = [
    ("agent_full_no_memory",    "full", ["--no-memory"]),
    ("agent_full_coarse",       "full", ["--coarse-report"]),
    ("agent_full_no_contract",  "full", ["--hide-contract"]),
    ("agent_full_no_escalate",  "full", ["--no-escalate"]),
]
RUN_ABLATIONS = False
if STAGE == "dev":
    SYSTEMS, RUN_ABLATIONS = SYSTEMS[:1], False
OUT = "/kaggle/working/results/agent_v1"
CACHE = "/kaggle/working/llm_cache"
print(STAGE, [s[0] for s in SYSTEMS], "ablations" if RUN_ABLATIONS else "")

dev ['agent_full'] 


In [2]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
NEEDED = ["eval/runner_agent.py", "src/igt/agents/repair_agent.py", "eval/agent_scenarios/rct_v2.jsonl",
          "eval/pool_v3/pool_v3.jsonl", "src/igt/llm/hf.py"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


commit f34d1e22ff920f02b704738d6db44d13d759b3d0


In [3]:
# ---------------- Libraries: pinned pandas for the sandbox + bitsandbytes ----------------
# The sandbox and verifier need pandas 2.3.3 / numpy 2.1.3 (pickles, dtype rules). Install them into a separate folder
# and use it only for this repo's processes, so Kaggle's torch stack stays untouched.
subprocess.run(["pip", "install", "-q", "--target", "/kaggle/working/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
if LOAD_IN_4BIT:
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH="/kaggle/working/evalpkgs")
p = subprocess.run(["python", "-c", "import pandas, numpy, torch; print(pandas.__version__, numpy.__version__, torch.__version__, torch.cuda.is_available())"],
                   env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
holidays 0.103 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 3.0.0 which is 

2.3.3 2.1.3 2.11.0+cu128 True
 


In [4]:
# ---------------- Run ----------------
def run(system, mode, flags, categories):
    cmd = ["python", "eval/runner_agent.py", "--system", system, "--mode", mode, "--backend", "hf",
           "--model", MODEL_ID, "--verifier", VERIFIER, "--splits", STAGE, "--categories", *categories,
           "--cache-dir", CACHE, "--out", OUT, *flags] + (["--load-in-4bit"] if LOAD_IN_4BIT else [])
    t0 = time.time()
    p = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    print(p.stdout[-2500:]); print(p.stderr[-2500:])
    assert p.returncode == 0, system
    print(f"== {system} done in {(time.time() - t0) / 60:.1f} min")

for system, mode, flags in SYSTEMS:
    run(system, mode, flags, ["R", "C", "T", "E", "S"])
if RUN_ABLATIONS:
    for system, mode, flags in ABLATIONS:
        run(system, mode, flags, ["R"])

T correct=None attempts=0 tools=0 (247s)
[seed 0] 13/46 R_dev_013 REJECT correct=None attempts=1 tools=0 (300s)
[seed 0] 14/46 R_dev_014 REJECT correct=None attempts=0 tools=0 (315s)
[seed 0] 15/46 C_dev_001 ACCEPT correct=True attempts=0 tools=0 (315s)
[seed 0] 16/46 C_dev_002 ACCEPT correct=True attempts=0 tools=0 (316s)
[seed 0] 17/46 C_dev_003 ACCEPT correct=True attempts=0 tools=0 (317s)
[seed 0] 18/46 C_dev_004 ACCEPT correct=True attempts=0 tools=0 (317s)
[seed 0] 19/46 C_dev_005 ACCEPT correct=True attempts=0 tools=0 (318s)
[seed 0] 20/46 C_dev_006 ACCEPT correct=True attempts=0 tools=0 (319s)
[seed 0] 21/46 C_dev_007 ACCEPT correct=True attempts=0 tools=0 (320s)
[seed 0] 22/46 C_dev_008 ACCEPT correct=True attempts=0 tools=0 (320s)
[seed 0] 23/46 C_dev_009 ACCEPT correct=True attempts=0 tools=0 (321s)
[seed 0] 24/46 C_dev_010 ACCEPT correct=True attempts=0 tools=0 (322s)
[seed 0] 25/46 C_dev_011 ACCEPT correct=True attempts=0 tools=0 (322s)
[seed 0] 26/46 C_dev_012 ACCEPT corr

In [5]:
# ---------------- Quick summary and zip ----------------
p = subprocess.run(["python", "eval/agent_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f"{OUT}/{d}"):
        open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)
        print("saved:", shutil.make_archive(f"/kaggle/working/agent_{d}_{STAGE}", "zip", f"{OUT}/{d}"))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)

system                     R succ  R esc  regr  C ok C unnec T recov gaming valid tool attempts  tokens unsafe E esc F1 S safe
agent_full                  0.000  0.071 0.000 1.000   0.000     -    0.000        -      0.020    2944      2    0.667  0.667
 
saved: /kaggle/working/agent_agent_full_dev.zip
